In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd
### example of pandas loading/saving:
# df.to_pickle("my_data.pkl") // SAVE
# df = pd.read_pickle("my_data.pkl") // LOAD
def get_workload_mix_features_dict(dict_location = '../data/python_data/',normalize = False):
    # loads dictionaries
    workload_filehandler = open(dict_location + 'workload_mixes.dict', 'rb') 
    workload_dict = pickle.load(workload_filehandler)
    jobtype_filehandler = open(dict_location + 'job_dictionary.dict', 'rb') 
    jobtype_dict = pickle.load(jobtype_filehandler)
    # for normalization
    if normalize:
        norm_weights = np.array([252.60714286, 346.34821429, 252.93928571, 297.96428571, 1.26071429,2.71428571])
    else:
        norm_weights = np.array([1, 1, 1, 1, 1, 1])
    # makes new dictionary
    work_feature_dict = {}
    for workload_mix in list(workload_dict.keys()):
        feature_list = []
        for jobtype in workload_dict[workload_mix]:
            # does normalization, calculated kinda manually 
            feature_list.append(jobtype_dict[jobtype]/norm_weights)
        
    
        work_feature_dict.update({workload_mix:np.array(feature_list)})
    return work_feature_dict
def convert_gd_to_new(gd_file_name,workload_mix_name,util_ratio,server_size,workload_dict_location):
    # finds useful quantities to know
    gd_file_df = pd.read_csv(gd_file_name)
    workload_mix_size = int((len(gd_file_df.columns)-10)/2)
    n_samples = len(gd_file_df.index)
    # grabs easy stuff from GD file
    cost_total = gd_file_df['CostTheory'].values
    cost_power = gd_file_df['costPower'].values
    cost_error = gd_file_df['costError'].values
    cost_qos = gd_file_df['costQoS'].values
    p = gd_file_df['P'].values
    r = gd_file_df['R'].values
    p_norm = gd_file_df['Pcoef'].values
    r_norm = gd_file_df['Rcoef'].values
    # makes columns for constants
    server_count = server_size * np.ones(n_samples)
    util = util_ratio * np.ones(n_samples)
    # grabs needed workload mix
    work_feature_dict = get_workload_mix_features_dict(workload_dict_location)
    workload_mix = work_feature_dict[workload_mix_name] # work_mix_size x 6 array 
    # grabs workload weights
    workload_weights = gd_file_df.iloc[:,10:10+workload_mix_size].values # n_samples x work_mix_size
    # combines them into final thing
    workload_mix_combined_list = []
    for sample_weights in workload_weights:
        workload_mix_combined = np.concatenate((workload_mix,np.array([sample_weights]).T),axis=1)
        workload_mix_combined_list.append(workload_mix_combined)
    data = {'cost':cost_total,
            'cost_power':cost_power,
            'cost_error':cost_error,
            'cost_qos':cost_qos,
            'p':p,
            'r':r,
            'p_norm':p_norm,
            'r_norm':r_norm,
            'client_count':server_count,
            'util':util,
            'workload_mix_size':workload_mix_size,
            'workload_mix':workload_mix_combined_list
            }
    new_df = pd.DataFrame(data)
    return new_df

In [5]:
### this is where we combine each GD file. 
# format is file,WL-mix,util,server size, dict location 
dict_list = []
## Experiment 10 
# W3
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util5.csv','W3',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util9_new.csv','W3',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util9.csv','W3',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util75_aggressive.csv','W3',0.75,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util75_conservative.csv','W3',0.75,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util75_new.csv','W3',0.75,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util99_cont.csv','W3',0.99,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util99_cont2.csv','W3',0.99,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w3-util99.csv','W3',0.99,500,''))
# W4
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w4-util5.csv','W4',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w4-util9.csv','W4',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w4-util75_alt_init.csv','W4',0.75,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w4-util75.csv','W4',0.75,500,''))
# W5 
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w5-util5.csv','W5',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w5-util5_cont.csv','W5',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w5-util9.csv','W5',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w5-util75.csv','W5',0.75,500,''))
# W6
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w6-util5.csv','W6',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w6-util75_bew.csv','W6',0.75,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w6-util9.csv','W6',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w6-util75.csv','W6',0.75,500,''))
# W7
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w7-util5.csv','W7',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w7-util9.csv','W7',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w7-util75.csv','W7',0.75,500,''))
# W8
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w8-util5.csv','W8',0.5,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w8-util9.csv','W8',0.9,500,''))
dict_list.append(convert_gd_to_new('exp_10_data/GDexp10-w8-util75.csv','W8',0.75,500,''))

## Experiment 11
dict_list.append(convert_gd_to_new('exp_11_data/GDexp11-gs-w3_util8.csv','W3',0.8,500,''))
dict_list.append(convert_gd_to_new('exp_11_data/GDexp11-gs-w4_util8.csv','W4',0.8,500,''))
dict_list.append(convert_gd_to_new('exp_11_data/GDexp11-gs-w5_util8.csv','W5',0.8,500,''))
dict_list.append(convert_gd_to_new('exp_11_data/GDexp11-gs-w6_util8.csv','W6',0.8,500,''))
dict_list.append(convert_gd_to_new('exp_11_data/GDexp11-gs-w7_util8.csv','W7',0.8,500,''))

## Experiment 12
dict_list.append(convert_gd_to_new('exp_12_data/GDexp12-gs-w3_util8_sc1000.csv','W3',0.8,1000,''))
dict_list.append(convert_gd_to_new('exp_12_data/GDexp12-gs-w4_util8_sc1000.csv','W4',0.8,1000,''))
dict_list.append(convert_gd_to_new('exp_12_data/GDexp12-gs-w5_util8_sc1000.csv','W5',0.8,1000,''))
dict_list.append(convert_gd_to_new('exp_12_data/GDexp12-gs-w6_util8_sc1000.csv','W6',0.8,1000,''))

## Experiment 13
dict_list.append(convert_gd_to_new('exp_13_data/GDexp13-w3.csv','W3',0.8,64,''))
dict_list.append(convert_gd_to_new('exp_13_data/GDexp13-w4.csv','W4',0.8,64,''))
dict_list.append(convert_gd_to_new('exp_13_data/GDexp13-w5.csv','W5',0.8,64,''))
dict_list.append(convert_gd_to_new('exp_13_data/GDexp13-w6.csv','W6',0.8,64,''))

## Experiment 14
dict_list.append(convert_gd_to_new('exp_14_data/GDexp14-a1.csv','A1',0.8,128,''))
dict_list.append(convert_gd_to_new('exp_14_data/GDexp14-a2.csv','A2',0.8,128,''))
dict_list.append(convert_gd_to_new('exp_14_data/GDexp14-b1.csv','B1',0.8,128,''))
dict_list.append(convert_gd_to_new('exp_14_data/GDexp14-b2.csv','B2',0.8,128,''))
dict_list.append(convert_gd_to_new('exp_14_data/GDexp14-b3.csv','B3',0.8,128,''))
dict_list.append(convert_gd_to_new('exp_14_data/GDexp14-c1.csv','C2',0.8,128,''))

# combines into one DF
final_data = pd.concat(dict_list, axis=0)
final_data
final_data.to_csv('all_data.csv', index=False)   

In [6]:
hour_sim_data = pd.read_csv('exp_11_data/GDexp11-gs-w3_util8.csv')

In [13]:
# loads workload mix dictionary because we'll need it obviously
work_feature_dict = get_workload_mix_features_dict('')
print(work_feature_dict['W3'])

[[237.   359.75  40.    53.     1.4    4.  ]
 [264.   399.   343.   381.     1.     4.  ]
 [250.   400.    89.   119.     1.4    9.  ]
 [242.   321.   165.   179.     1.6    6.  ]
 [253.   370.   329.   352.     1.2    8.  ]
 [246.   336.   231.   242.     0.9    6.  ]
 [328.   366.    36.    49.     1.5    4.  ]
 [228.   287.    27.3   27.7    2.     4.  ]]


In [37]:
thing = np.array([[1,2,3],[4,5,6]])
print(thing)
weight = np.array([[8,9]])
print(weight.T)

[[1 2 3]
 [4 5 6]]
[[8]
 [9]]


In [39]:
print(np.concatenate((thing,weight.T),axis=1))

[[1 2 3 8]
 [4 5 6 9]]
